<a href="https://colab.research.google.com/github/Rds1007/SQL_BigDataInterview/blob/main/LatestRecordBasedOnSalaryHistory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Finding Updated Records

The difficulty is Easy, because the task requires aggregating data to determine the most recent salary — this involves grouping records by multiple fields to correctly identify individual employees, and then applying an operation to extract the maximum salary value for each group. The logic is not complex but requires understanding of data grouping and applying basic summary operations across different programming language implementations.


We have a table with employees and their salaries; however, some of the records are old and contain outdated salary information. Since there is no timestamp, assume salary is non-decreasing over time. You can consider the current salary for an employee is the largest salary value among their records. If multiple records share the same maximum salary, return any one of them. Output their id, first name, last name, department ID, and current salary. Order your list by employee ID in ascending order.

In [1]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import SparkSession


In [3]:
spark=SparkSession.builder.appName('practice').getOrCreate()

--select * from ms_employee_salary;

with cre as
(
select id,first_name,last_name,salary,department_id,
ROW_NUMBER() over(partition by first_name,last_name  order by salary desc) as rnk
from ms_employee_salary
)
select id, first_name,last_name, department_id,salary from cre where rnk=1 order by id asc

In [9]:
df=spark.read.csv('/content/sample_data/ms_employee_salary.csv',header=True, inferSchema=True)

In [10]:
df.show()

+---+----------+---------+------+-------------+
| id|first_name|last_name|salary|department_id|
+---+----------+---------+------+-------------+
|  1|      Todd|   Wilson|110000|         1006|
|  1|      Todd|   Wilson|106119|         1006|
|  2|    Justin|    Simon|128922|         1005|
|  2|    Justin|    Simon|130000|         1005|
|  3|     Kelly|  Rosario| 42689|         1002|
|  4|  Patricia|   Powell|162825|         1004|
|  4|  Patricia|   Powell|170000|         1004|
|  5|    Sherry|   Golden| 44101|         1002|
|  6|   Natasha|  Swanson| 79632|         1005|
|  6|   Natasha|  Swanson| 90000|         1005|
|  7|     Diane|   Gordon| 74591|         1002|
|  8|  Mercedes|Rodriguez| 61048|         1005|
|  9|   Christy| Mitchell|137236|         1001|
|  9|   Christy| Mitchell|140000|         1001|
|  9|   Christy| Mitchell|150000|         1001|
| 10|      Sean| Crawford|182065|         1006|
| 10|      Sean| Crawford|190000|         1006|
| 11|     Kevin| Townsend|166861|       

In [11]:
window=Window.partitionBy('first_name','last_name').orderBy(desc('salary'))

In [14]:
df = df.withColumn("rank", row_number().over(window)).filter(col("rank") == 1).drop("rank").orderBy("id")

In [15]:
df.show()

+---+----------+---------+------+-------------+
| id|first_name|last_name|salary|department_id|
+---+----------+---------+------+-------------+
|  1|      Todd|   Wilson|110000|         1006|
|  2|    Justin|    Simon|130000|         1005|
|  3|     Kelly|  Rosario| 42689|         1002|
|  4|  Patricia|   Powell|170000|         1004|
|  5|    Sherry|   Golden| 44101|         1002|
|  6|   Natasha|  Swanson| 90000|         1005|
|  7|     Diane|   Gordon| 74591|         1002|
|  8|  Mercedes|Rodriguez| 61048|         1005|
|  9|   Christy| Mitchell|150000|         1001|
| 10|      Sean| Crawford|190000|         1006|
| 11|     Kevin| Townsend|166861|         1002|
| 12|    Joshua|  Johnson|123082|         1004|
| 13|     Julie|  Sanchez|210000|         1001|
| 14|      John|  Coleman|152434|         1001|
| 15|   Anthony|   Valdez| 96898|         1001|
| 16|    Briana|    Rivas|151668|         1005|
| 17|     Jason|  Burnett| 42525|         1006|
| 18|   Jeffrey|   Harris| 20000|       